# 第 5 週 實作｜三角積分與三角代換

前半處理「被積式本來就是三角函數」,後半更神奇:被積式明明沒有三角函數,卻要硬塞一個進去——只為了讓恆等式把根號吃掉。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜SymPy 當裁判:手算對不對

這週的題目手算容易錯一個負號就全毀。用 SymPy 當標準答案,但重點是<strong>看懂它給的形式和你的形式為什麼等價</strong>。


In [ ]:
x = sp.Symbol('x', real=True)

cases = [
    ("∫ sin^3 x cos^2 x dx",  sp.sin(x)**3 * sp.cos(x)**2,
     sp.cos(x)**5/5 - sp.cos(x)**3/3),
    ("∫ sin^2 x dx",          sp.sin(x)**2,
     x/2 - sp.sin(2*x)/4),
    ("∫ sin^2 x cos^2 x dx",  sp.sin(x)**2*sp.cos(x)**2,
     x/8 - sp.sin(4*x)/32),
    ("∫ tan^2 x dx",          sp.tan(x)**2,
     sp.tan(x) - x),
    ("∫ sqrt(1-x^2) dx",      sp.sqrt(1-x**2),
     sp.asin(x)/2 + x*sp.sqrt(1-x**2)/2),
    ("∫ dx/sqrt(x^2+1)",      1/sp.sqrt(x**2+1),
     sp.log(x + sp.sqrt(x**2+1))),
]

print(f"{'積分':28s} {'我的答案微分回去 == 被積式?':>28}")
for name, f, mine in cases:
    ok = sp.simplify(sp.diff(mine, x) - f) == 0
    print(f"{name:28s} {str(ok):>28}")

print("\n注意 SymPy 給的形式可能和課本不同,但等價:")
ref = sp.integrate(sp.sin(x)**3*sp.cos(x)**2, x)
mine = sp.cos(x)**5/5 - sp.cos(x)**3/3
print("  SymPy:", ref)
print("  課本 :", mine)
print("  差為常數?", sp.simplify(sp.diff(ref - mine, x)) == 0)

In [ ]:
# TODO 學生練習:把你手算的 ∫ cos^5 x dx 填進來,用同樣方法驗證
# mine = ???
# print(sp.simplify(sp.diff(mine, x) - sp.cos(x)**5) == 0)

## Lab 2｜三角代換的定義域陷阱

觀念 7 說值域限制是「拿掉絕對值的許可證」。這格用數值方法看破壞規則會發生什麼事。


In [ ]:
# sqrt(x^2 - 1) = |tan(theta)|,只有在對的分支才等於 tan(theta)
print("x = sec(theta) 的兩支:")
for theta_deg in [30, 60, 120, 150]:
    th = math.radians(theta_deg)
    xv = 1/math.cos(th)                     # x = sec(theta)
    lhs = math.sqrt(xv**2 - 1)              # sqrt(x^2 - 1) 一定 >= 0
    rhs = math.tan(th)                      # tan(theta) 可能為負
    print(f"  theta={theta_deg:4d}deg  x={xv:7.3f}  sqrt(x^2-1)={lhs:6.3f}  "
          f"tan(theta)={rhs:7.3f}  相等? {abs(lhs-rhs) < 1e-12}")

print("\n→ theta > 90 度(對應 x < -1)時 tan 是負的,不能直接寫 sqrt = tan")

# 定積分換限:限也必須落在允許區間
x = sp.Symbol('x', real=True)
th = sp.Symbol('theta', real=True)
exact = sp.integrate(sp.sqrt(1 - x**2), (x, 0, 1))
# 換元:x = sin(theta),限 0 -> pi/2
subbed = sp.integrate(sp.cos(th)**2, (th, 0, sp.pi/2))
print(f"\n∫_0^1 sqrt(1-x^2) dx = {exact}  (= pi/4 = 四分之一圓)")
print(f"換元後 ∫_0^(pi/2) cos^2 = {subbed}  一致? {sp.simplify(exact - subbed) == 0}")

# 值域蓋不住會怎樣:x = sin(theta) 處理 |x| <= 2 是不行的
print("\nx = sin(theta) 的值域 = [-1, 1];要處理 sqrt(4-x^2) 必須用 x = 2 sin(theta)")